# Cantilever: **pyGmsh brick**–Frame Coupling visualised in pyGmshViewer

Same coupling recipe as `example_cantilever_brick_frame_coupling_viewer.ipynb`
but now the brick mesh is generated by **pyGmsh**. Workflow:

1. Build an axis-aligned box with `g.model.geometry.add_box(...)`
2. Set transfinite + recombine on every curve / face / volume so gmsh
   produces a structured **hexahedral** mesh
3. Extract a `FEMData` object (solver-ready contiguous IDs, RCM reorder)
4. Build the OpenSees brick model from `fem.node_ids / connectivity`
5. Identify interface face nodes by coordinate (x ≈ L_solid) and attach
   duplicated nodes + `rigidLink('beam')` + `equalDOF` exactly as before
6. Solve + export solid / frame / rigid-links to three VTU files
7. Launch the viewer (frame + spokes now render as thick tubes)

## 1. Parameters

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import gmsh
import openseespy.opensees as ops
from apeGmsh import apeGmsh

# Geometry [m]
L       = 10.0
h       = 1.0       # y-depth
b       = 0.5       # z-width
L_solid = L / 2.0
L_frame = L / 2.0

# Material
E, nu = 30.0e6, 0.2
G = E / (2.0 * (1.0 + nu))

# Loading
P = -100.0          # tip load [kN], -y

# Mesh density (structured hex)
nx_s, ny_s, nz_s = 20, 6, 3

# Frame section
A  = b * h
Iz = b * h**3 / 12.0
Iy = h * b**3 / 12.0
J  = 0.5 * (Iy + Iz)

print(f'Box: {L_solid} x {h} x {b} m  |  hex grid {nx_s}x{ny_s}x{nz_s}')

## 2. Build the hex mesh with pyGmsh

`pyGmsh` wraps the OCC kernel. We set transfinite constraints by
direction so the mesh has exactly `nx_s × ny_s × nz_s` hex cells.

In [ ]:
g = apeGmsh(model_name='CantileverBrick', verbose=False)
g.begin()

# Box: origin at (0, -h/2, -b/2), centred on the beam axis
box_tag = g.model.geometry.add_box(0.0, -h/2.0, -b/2.0, L_solid, h, b, label='solid_block')

# Assign transfinite node counts to every curve based on its direction
def _curve_dir(ctag, tol=1e-6):
    xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.getBoundingBox(1, ctag)
    dx, dy, dz = xmax - xmin, ymax - ymin, zmax - zmin
    if dx > tol and dy < tol and dz < tol: return 'x'
    if dy > tol and dx < tol and dz < tol: return 'y'
    if dz > tol and dx < tol and dy < tol: return 'z'
    return '?'

nodes_per_dir = {'x': nx_s + 1, 'y': ny_s + 1, 'z': nz_s + 1}
for (_, ctag) in gmsh.model.getEntities(1):
    gmsh.model.mesh.setTransfiniteCurve(ctag, nodes_per_dir[_curve_dir(ctag)])

# Transfinite + recombine on every face, transfinite volume
for (_, stag) in gmsh.model.getEntities(2):
    gmsh.model.mesh.setTransfiniteSurface(stag)
    gmsh.model.mesh.setRecombine(2, stag)
for (_, vtag) in gmsh.model.getEntities(3):
    gmsh.model.mesh.setTransfiniteVolume(vtag)

g.mesh.generation.generate(dim=3)

# Renumber to contiguous solver-ready IDs with RCM bandwidth optimisation
g.mesh.partitioning.renumber_mesh(dim=3, method='rcm', base=1)
fem = g.mesh.queries.get_fem_data(dim=3)
print(fem.info)

## 3. Identify the interface face (x ≈ L_solid)

The coupling plane is the face at x = L_solid. We pick every solid
node whose x-coordinate lies within a small tolerance of that plane.

In [ ]:
tol = 1.0e-6
x_coords = fem.node_coords[:, 0]
interface_mask = np.abs(x_coords - L_solid) < tol
interface_solid_tags = fem.node_ids[interface_mask].astype(int).tolist()

# Also find the fixed-support face at x = 0 (cantilever base)
base_mask = np.abs(x_coords - 0.0) < tol
base_solid_tags = fem.node_ids[base_mask].astype(int).tolist()

print(f'Base nodes      (x = 0)       : {len(base_solid_tags)}')
print(f'Interface nodes (x = {L_solid}) : {len(interface_solid_tags)}')
print(f'Expected        (ny+1)*(nz+1) = {(ny_s+1)*(nz_s+1)}')

## 4. Build the OpenSees model

The solid block is created in an `ndf=3` builder because `stdBrick`
uses translational DOFs only. We then switch to `ndf=6` for the
master / tip / dup nodes so `rigidLink('beam')` and
`elasticBeamColumn` have the rotational DOFs they need.

In [ ]:
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 3)
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

# -- Nodes from the pyGmsh-generated mesh --
for i in range(fem.info.n_nodes):
    ops.node(int(fem.node_ids[i]), *fem.node_coords[i])

# Cantilever base: translational fixity at x = 0
for nid in base_solid_tags:
    ops.fix(int(nid), 1, 1, 1)

# -- Brick elements -- fem.connectivity is already in solver IDs
for i in range(fem.info.n_elems):
    eid   = int(fem.element_ids[i])
    nodes = [int(n) for n in fem.connectivity[i]]
    ops.element('stdBrick', eid, *nodes, 1)
print(f'Created {fem.info.n_elems} stdBrick elements')

## 5. Master / tip / dup nodes + coupling

In [ ]:
# Switch to 6 DOFs for frame + coupling nodes
ops.model('basic', '-ndm', 3, '-ndf', 6)

max_sid = int(fem.node_ids.max())
master_tag = max_sid + 1
tip_tag    = max_sid + 2
ops.node(master_tag, L_solid, 0.0, 0.0)
ops.node(tip_tag,    L,       0.0, 0.0)

# Duplicated nodes at each interface face node
dup_tags = []
next_id = tip_tag + 1
for sn in interface_solid_tags:
    c = ops.nodeCoord(int(sn))
    ops.node(next_id, *c)
    dup_tags.append(next_id)
    next_id += 1

# Coupling: master -> dup (rigidLink 'beam')  +  dup -> solid (equalDOF translations)
for dn in dup_tags:
    ops.rigidLink('beam', master_tag, dn)
for dn, sn in zip(dup_tags, interface_solid_tags):
    ops.equalDOF(dn, int(sn), 1, 2, 3)

# Frame element
ops.geomTransf('Linear', 1, 0.0, 0.0, 1.0)
frame_eid = int(fem.element_ids.max()) + 1
ops.element('elasticBeamColumn', frame_eid, master_tag, tip_tag,
            A, E, G, J, Iy, Iz, 1)
print(f'master={master_tag}  tip={tip_tag}  dup={dup_tags[0]}..{dup_tags[-1]}')
print(f'frame element tag = {frame_eid}')

## 6. Loading & analysis

In [ ]:
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
ops.load(tip_tag, 0.0, float(P), 0.0, 0.0, 0.0, 0.0)

# Chained MPCs (master -> dup -> solid) are handled robustly with Penalty
ops.constraints('Penalty', 1.0e12, 1.0e12)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-8, 30, 0)
ops.algorithm('Linear')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')

ok = ops.analyze(1)
print('analyze ok =', ok)
print(f'tip uy = {ops.nodeDisp(tip_tag, 2):.4e} m')

---
## 7. Viewer export

Same three-VTU export: brick volume, frame line, rigid-link spokes.

In [ ]:
# 7.1 -- Brick mesh + displacement --
N = fem.info.n_nodes
solid_points = np.asarray(fem.node_coords, dtype=np.float64)
solid_disp   = np.zeros((N, 3))
for i in range(N):
    d = ops.nodeDisp(int(fem.node_ids[i]))
    solid_disp[i] = d[:3]

# fem.connectivity uses solver IDs (1-based RCM ordering); we need 0-based
# row indices into fem.node_ids.
sid_to_row = {int(sid): r for r, sid in enumerate(fem.node_ids)}
solid_cells = np.array(
    [[sid_to_row[int(n)] for n in row] for row in fem.connectivity],
    dtype=np.int32,
)
print(f'Solid hex  : {N} nodes, {len(solid_cells)} cells')
print(f'  |u| max  : {np.linalg.norm(solid_disp, axis=1).max():.4e} m')

In [ ]:
# 7.2 -- Frame line + rigid-link spokes --
master_xyz = np.array(ops.nodeCoord(master_tag)); master_d = np.array(ops.nodeDisp(master_tag))[:3]
tip_xyz    = np.array(ops.nodeCoord(tip_tag));    tip_d    = np.array(ops.nodeDisp(tip_tag))[:3]

frame_points = np.vstack([master_xyz, tip_xyz])
frame_disp   = np.vstack([master_d, tip_d])
frame_cells  = np.array([[0, 1]], dtype=np.int32)

n_dup = len(dup_tags)
links_points = np.zeros((1 + n_dup, 3))
links_disp   = np.zeros((1 + n_dup, 3))
links_points[0] = master_xyz
links_disp[0]   = master_d
for k, dn in enumerate(dup_tags):
    links_points[k+1] = ops.nodeCoord(dn)
    links_disp[k+1]   = ops.nodeDisp(dn)[:3]
links_cells = np.array([[0, k+1] for k in range(n_dup)], dtype=np.int32)

print(f'Frame: 1 line  |  Rigid links: {n_dup} spokes from master')

In [ ]:
# 7.3 -- Write VTU files --
from pathlib import Path
from apeGmsh.viz.VTKExport import write_vtu

VTK_HEXAHEDRON = 12
VTK_LINE       = 3

solid_vtu = Path('pygmsh_brick_solid.vtu').resolve()
frame_vtu = Path('pygmsh_brick_frame.vtu').resolve()
links_vtu = Path('pygmsh_brick_rigid_links.vtu').resolve()

write_vtu(
    solid_vtu, points=solid_points, cells=solid_cells,
    vtk_cell_type=VTK_HEXAHEDRON,
    point_data={
        'Displacement': solid_disp,
        '|U|':          np.linalg.norm(solid_disp, axis=1),
        'Ux':           solid_disp[:, 0],
        'Uy':           solid_disp[:, 1],
        'Uz':           solid_disp[:, 2],
    },
)
write_vtu(frame_vtu, points=frame_points, cells=frame_cells,
          vtk_cell_type=VTK_LINE,
          point_data={'Displacement': frame_disp,
                      '|U|': np.linalg.norm(frame_disp, axis=1)})
write_vtu(links_vtu, points=links_points, cells=links_cells,
          vtk_cell_type=VTK_LINE,
          point_data={'Displacement': links_disp,
                      '|U|': np.linalg.norm(links_disp, axis=1)})
print('Wrote:')
for p in (solid_vtu, frame_vtu, links_vtu):
    print(' ', p.name)

g.end()

In [ ]:
# 7.4 -- Launch pyGmshViewer --
from apeGmshViewer import show

show(solid_vtu, frame_vtu, links_vtu, blocking=False)
print('Viewer launched. Try:')
print('  1. Select pygmsh_brick_solid -> Contour |U|, Deformation 50x')
print('  2. The amber line (frame) and spokes (rigid links) render as')
print('     thick tubes; each spoke rotates about the master node')
print('  3. Ctrl+P on an interface brick face node: its displacement')
print('     matches the dup node at the same position (equalDOF 1,2,3)')